# 03 - Modelagem

Notebook para treinar, validar e comparar modelos preditivos de risco de credito.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

def plot_lorenz_gini(y_true, y_score, title=''):
    '''
    Plot the Lorenz/CAP curve and calculate the Gini coefficient.
    y_true: actual values (0/1)
    y_score: probability predicted by the model
    '''
    # Work on plain arrays so misaligned Series indices can't introduce NaNs
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n_pos = y_true.sum()
    if n_pos == 0 or n_pos == len(y_true):
        raise ValueError('y_true must contain both positive and negative cases.')

    # Sort by score in descending order (riskiest first)
    df_sort = pd.DataFrame({'real': y_true, 'score': y_score})
    df_sort = df_sort.sort_values('score', ascending=False).reset_index(drop=True)

    # Cumulative proportion of delinquent accounts identified vs. population
    df_sort['cum_real'] = df_sort['real'].cumsum() / n_pos
    df_sort['cum_pop'] = (df_sort.index + 1) / len(df_sort)

    # Gini = 2 * AUC_ROC - 1 (the 2*AUC-1 identity holds for the ROC curve,
    # not for the cumulative-gains curve plotted below).
    gini = 2 * roc_auc_score(y_true, y_score) - 1

    # Plot the Lorenz/CAP curve
    p = n_pos / len(y_true)
    plt.figure(figsize=(7, 6))
    plt.plot(
        df_sort['cum_pop'],
        df_sort['cum_real'],
        color='#2E5EAA',
        linewidth=2,
        label=f'Model (Gini={gini:.3f})'
    )
    # Perfect model captures all positives within the first p of the population
    plt.plot(
        [0, p, 1],
        [0, 1, 1],
        color='#E65100',
        linestyle=':',
        label='Perfect model (Gini=1)'
    )
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random (Gini=0)')
    plt.xlabel('Cumulative proportion of the population')
    plt.ylabel('Cumulative proportion of delinquent borrowers')
    plt.title(f'Curva de Lorenz — {title}', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return gini

In [ ]:
# Permite importar os módulos do projeto (pasta src/ na raiz do repositório)
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.preprocessing import tratar_nulos
from src.features import criar_features, codificar_categoricas

from sklearn.model_selection import train_test_split, StratifiedKFold
from imblearn.over_sampling import SMOTENC
import lightgbm as lgb

In [ ]:
# Prepara os dados (reutiliza as funções dos módulos em src/)
df = pd.read_csv('../data/raw/application_train.csv')

# Separa features (X) da variável-alvo (y) — ainda com nulos e colunas object
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# Divisão: 80% treino, 20% teste
# stratify=y garante a mesma proporção de inadimplentes em ambos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Todo o pré-processamento abaixo APRENDE apenas no treino e é reaplicado ao teste,
# evitando data leakage (imputação, criação de features e codificação).
X_train, imputers = tratar_nulos(X_train, fit=True)
X_test = tratar_nulos(X_test, fit=False, imputers=imputers)

X_train = criar_features(X_train)
X_test = criar_features(X_test)

X_train, encoders = codificar_categoricas(X_train, fit=True)
X_test = codificar_categoricas(X_test, fit=False, encoders=encoders)

# Índices das colunas categóricas (já codificadas) — usados pelo SMOTENC
cat_idx = [X_train.columns.get_loc(col) for col in encoders]

print(f'Treino: {X_train.shape[0]:,} amostras')
print(f'Teste:  {X_test.shape[0]:,} amostras')
print(f'Taxa inadimplência treino: {y_train.mean():.3f}')

In [ ]:
# SMOTENC: cria amostras sintéticas da classe minoritária (inadimplentes).
# Diferente do SMOTE, o SMOTENC trata as colunas categóricas corretamente
# (não interpola códigos — escolhe a categoria mais frequente entre os vizinhos).
# IMPORTANTE: aplicar apenas no treino, nunca no teste.
smote = SMOTENC(categorical_features=cat_idx, random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Após SMOTENC — Adimplentes:    {(y_train_bal == 0).sum():,}')
print(f'Após SMOTENC — Inadimplentes: {(y_train_bal == 1).sum():,}')

In [ ]:
# Otimização de hiperparâmetros com Optuna
import optuna
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    '''
    Função que o Optuna vai tentar maximizar (AUC).
    Cada 'trial' experimenta uma combinação diferente de hiperparâmetros.
    '''
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        # Optuna escolhe os valores abaixo automaticamente:
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    # Validação cruzada estratificada (5 folds) sobre os dados ORIGINAIS de treino.
    # O SMOTENC é aplicado dentro de cada fold, somente no subconjunto de treino,
    # para que a validação seja feita em dados reais (sem data leakage).
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_scores = []

    for idx_tr, idx_val in cv.split(X_train, y_train):
        X_tr = X_train.iloc[idx_tr]
        X_val = X_train.iloc[idx_val]
        y_tr = y_train.iloc[idx_tr]
        y_val = y_train.iloc[idx_val]

        # Balanceia apenas o treino do fold
        X_tr, y_tr = SMOTENC(
            categorical_features=cat_idx, random_state=42, k_neighbors=5
        ).fit_resample(X_tr, y_tr)

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        y_pred = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, y_pred))

    return np.mean(auc_scores)


# Executa 50 trials (pode aumentar para melhor resultado)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Melhor AUC: {study.best_value:.4f}')
print(f'Melhores parâmetros: {study.best_params}')

In [ ]:
# Modelo final: treina com os melhores hiperparâmetros, avalia no TESTE e salva
import joblib
import json
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from src.scoring import adjust_probability_to_prior, probability_to_scorecard_points

# Parâmetros fixos + os melhores encontrados pelo Optuna
# (merge cria um novo dict — não muta study.best_params)
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    **study.best_params,
}

# Treina no treino balanceado por SMOTENC (X_train_bal/y_train_bal já estão codificados)
modelo_final = lgb.LGBMClassifier(**best_params)
modelo_final.fit(X_train_bal, y_train_bal)

# Avaliação no conjunto de teste (dados reais, nunca vistos e não balanceados)
y_prob = modelo_final.predict_proba(X_test)[:, 1]

# Métricas principais
auc = roc_auc_score(y_test, y_prob)
gini = 2 * auc - 1  # Gini = 2*AUC - 1 (identidade válida para a curva ROC)

# KS (Kolmogorov-Smirnov): separação máxima entre as distribuições de bons e maus
ks_stat, _ = ks_2samp(y_prob[y_test == 0], y_prob[y_test == 1])

print(f'AUC-ROC: {auc:.4f}')
print(f'KS:      {ks_stat:.4f}')
print(f'Gini:    {gini:.4f}')

# Referências da indústria:
# AUC > 0.70 = aceitável | > 0.75 = bom | > 0.80 = muito bom
# KS  > 0.30 = aceitável | > 0.40 = bom | > 0.50 = muito bom

# Curva de Lorenz / Gini (função definida no início do notebook)
plot_lorenz_gini(y_test, y_prob, title='Modelo Final (teste)')

# Salva os artefatos necessários para escorar novos dados:
# modelo + imputers (medianas) + encoders reproduzem o pré-processamento.
joblib.dump(modelo_final, '../models/modelo_credito.pkl')
joblib.dump(imputers, '../models/imputers.pkl')
joblib.dump(encoders, '../models/encoders.pkl')

# Configuração de scoring consumida pelo app (dashboard/app.py): converte a PD do
# modelo campeão em score (300–850) e faixas. A correção de prior remove o viés do
# treino balanceado; PDO=30 evita saturação; cortes de faixa por tercil no teste.
prior = float(y_train.mean())
odds_ref = (1 - prior) / prior
PDO = 30
pd_real_test = adjust_probability_to_prior(y_prob, target_prior=prior)
scores_test = probability_to_scorecard_points(pd_real_test, pdo=PDO, odds_ref=odds_ref, score_ref=600)
c_high, c_low = [float(x) for x in np.quantile(scores_test, [1 / 3, 2 / 3])]

scoring_config = {
    'prior': prior,
    'pdo': PDO,
    'odds_ref': float(odds_ref),
    'score_ref': 600,
    'band_medium_min': round(c_high, 1),
    'band_low_min': round(c_low, 1),
    'raw_feature_order': list(X.columns),
}
with open('../models/scoring_config.json', 'w', encoding='utf-8') as f:
    json.dump(scoring_config, f, indent=2)

print('Artefatos salvos em ../models/: modelo_credito.pkl, imputers.pkl, encoders.pkl, scoring_config.json')

In [ ]:
# Importância das variáveis (top 20) — quanto cada feature contribuiu para o modelo
importancias = (
    pd.DataFrame({
        'feature': X_train_bal.columns,
        'importance': modelo_final.feature_importances_,
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

top20 = importancias.head(20).iloc[::-1]  # invertido para o maior ficar no topo do gráfico

plt.figure(figsize=(8, 7))
plt.barh(top20['feature'], top20['importance'], color='#2E5EAA')
plt.xlabel('Importância (ganho/splits)')
plt.title('Top 20 variáveis mais importantes', fontweight='bold')
plt.tight_layout()
plt.show()

importancias.head(20)

## 6 — Interpretabilidade com SHAP

Modelos de crédito precisam ser **explicáveis** (exigência regulatória e de negócio). O SHAP atribui a cada feature sua contribuição para cada previsão, permitindo entender o modelo globalmente e justificar decisões individuais.

In [ ]:
# Importância global das features com SHAP
import shap

# Explicador SHAP para o modelo de árvore (LightGBM)
explainer = shap.TreeExplainer(modelo_final)

# Amostra do teste (usar todas as linhas pode ser lento; 1000 basta para exploração)
X_sample = X_test.sample(1000, random_state=42)
shap_values = explainer.shap_values(X_sample)

# Compatibilidade entre versões do SHAP: versões antigas retornam uma lista
# [classe_0, classe_1]; as novas (0.45+) já retornam o array da classe positiva.
shap_vals_inadim = shap_values[1] if isinstance(shap_values, list) else shap_values

# Gráfico de importância global (bar plot)
shap.summary_plot(shap_vals_inadim, X_sample, plot_type='bar',
                  max_display=20, show=False, plot_size=(10, 8))
plt.title('Top 20 Features — Importância Global (SHAP)', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/shap_importancia_global.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 — Beeswarm: impacto e direção

Enquanto o gráfico de barras mostra apenas a magnitude média, o beeswarm revela **a direção** do efeito de cada feature: para que lado (aumenta/reduz risco) e em função de valores altos (vermelho) ou baixos (azul).

In [ ]:
# Beeswarm: mostra COMO cada feature afeta a previsão
# Cor vermelha = valor alto da feature, azul = valor baixo
# Eixo X: negativo = reduz risco, positivo = aumenta risco
shap.summary_plot(shap_vals_inadim, X_sample, max_display=15, show=False, plot_size=(10, 8))
plt.title('SHAP Beeswarm — Direção do Impacto das Features', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

# Como ler: EXT_SOURCE_2 com valor ALTO (vermelho, direita) = REDUZ risco;
# com valor BAIXO (azul, esquerda) = AUMENTA risco.

### 6.3 — Explicação individual de um cliente

O waterfall mostra como cada feature empurrou a previsão deste cliente acima ou abaixo do valor base (esperado). É a visão usada por compliance/áreas de negócio para justificar uma decisão de crédito caso a caso.

In [ ]:
# Explicação individual de UM cliente (o que um analista de compliance veria)
cliente_idx = 42  # qualquer índice posicional do teste
cliente = X_test.iloc[[cliente_idx]]
score = modelo_final.predict_proba(cliente)[0, 1]

print(f'Score de risco do cliente: {score:.3f}')
print(f'Decisão: {"REPROVAR" if score > 0.5 else "APROVAR"}')

# Waterfall: contribuição de cada feature para a previsão deste cliente
shap_cliente = explainer(cliente)
exp_individual = shap_cliente[0]
# Compatibilidade: se vier um eixo extra de classes, seleciona a classe 1 (inadimplência)
if exp_individual.values.ndim == 2:
    exp_individual = exp_individual[:, 1]
shap.plots.waterfall(exp_individual, max_display=12, show=False)
plt.tight_layout()
plt.savefig('../reports/figures/shap_waterfall_cliente.png', dpi=150, bbox_inches='tight')
plt.show()